In [17]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [18]:
df = pd.read_csv(('banknote_authentication.csv'), sep = ',')

x = df.drop('class', axis = 1).values
y = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state=42)

standardScaler = StandardScaler()
X_train_scaled = standardScaler.fit_transform(X_train)
y_train = y_train
X_test_scaled = standardScaler.transform(X_test)

In [19]:
class DecisionTree:
  class Node:
      def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

      def is_leaf(self):
        if self.value is not None:
          return True
        else:
          return False

  def __init__(self):
    self.root = None
    self.best_gain = float("inf")
    self.split_index = None
    self.split_threshold = None
    self.best_feature = None
    self.best_threshold = None

  def fit(self, X, y):
    self.root = self.build_tree(X, y)

  def Most_Common_Label(self, y):
      labels, counts = np.unique(y, return_counts=True)
      index = np.argmax(counts)
      return labels[index]

  def Split(self, X_column, split_threshold):
    left_indices = np.argwhere(X_column <= split_threshold).flatten()
    right_indices = np.argwhere(X_column > split_threshold).flatten()
    return left_indices, right_indices

  def Entropy(self, y):
    fID3 = np.mean(y)
    if fID3 == 0 or fID3 == 1:
      return 0
    else:
      return -fID3 * np.log(fID3) - (1 - fID3) * np.log(1 - fID3)

  def Information_Gain(self, y, X_column, split_threshold):
    left_indices, right_indices = self.Split(X_column, split_threshold)
    if len(left_indices) == 0 or len(right_indices) == 0:
      return float('inf')
    e_left = self.Entropy(y[left_indices])
    e_right = self.Entropy(y[right_indices])
    nl = len(left_indices)
    nr = len(right_indices)
    n = len(y)
    weighted_entropy = (nl / n) * e_left + (nr / n) * e_right
    return weighted_entropy

  def Best_Split(self, X, y):
    best_gain = float("inf")
    split_index = None
    split_threshold = None

    for feature_index in range(X.shape[1]):
      X_column = X[:, feature_index]
      X_column_sorted = np.sort(X_column)
      thresholds = (X_column_sorted[:-1] + X_column_sorted[1:])/2
      for threshold in thresholds:
        gain = self.Information_Gain(y, X_column, threshold)
        if gain < best_gain:
          best_gain = gain
          split_index = feature_index
          split_threshold = threshold
    return split_index, split_threshold


  def build_tree(self, X, y):
    n_labels = len(np.unique (y))
    if (n_labels == 1):
      leaf_value = self.Most_Common_Label(y)
      return self.Node(value=leaf_value)

    best_feature, best_threshold = self.Best_Split(X, y)

    if best_feature is None:
        leaf_value = self.Most_Common_Label(y)
        return self.Node(value=leaf_value)

    left_indices, right_indices = self.Split(X[:, best_feature], best_threshold)

    if len(left_indices) == 0 or len(right_indices) == 0:
        leaf_value = self.Most_Common_Label(y)
        return self.Node(value=leaf_value)


    left_subtree = self.build_tree(X[left_indices, :], y[left_indices])
    right_subtree = self.build_tree(X[right_indices, :], y[right_indices])
    return self.Node(best_feature, best_threshold, left_subtree, right_subtree)


  def traverse_tree(self, x, node):
    if node.is_leaf():
      return node

    if x[node.feature] <= node.threshold:
      return self.traverse_tree(x, node.left)
    else:
      return self.traverse_tree(x, node.right)

  def predict(self, X):
    predictions = []
    for x_row in X:
      predicted_node = self.traverse_tree(x_row, self.root)
      predictions.append(predicted_node.value)
    return np.array(predictions)

class RandomForest:
  def __init__(self, n_estimators = 100):
    self.n_estimators = n_estimators
    self.trees = []

  def fit(self, X, y):
    self.trees = []
    for i in range(self.n_estimators):
      tree = DecisionTree()
      X_sample, y_sample = self.bootstrap_sample(X, y)
      tree.fit(X_sample, y_sample)
      self.trees.append(tree)


  def predict(self, X):
    predictions = []
    for tree in self.trees:
      prediction = tree.predict(X)
      predictions.append(prediction)
    predictions = np.array(predictions)
    predictions = np.swapaxes(predictions, 0, 1)
    preds = []
    for prediction in predictions:
      pred = self.Most_Common_Label(prediction)
      preds.append(pred)
    return np.array(preds)

  def Most_Common_Label(self, y):
    labels, counts = np.unique(y, return_counts=True)
    index = np.argmax(counts)
    return labels[index]

  def bootstrap_sample(self, X, y):
    num_samples, num_features = X.shape
    indices = np.random.choice(num_samples, num_samples, replace = True)
    return X[indices], y[indices]

In [20]:
print("--- Random Forest using NumPy ---")

model = RandomForest()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print("Prediction:", y_pred.flatten()[:20])
print("y values:", y_test[:20])

def accuracy(y, y_pred):
  accuracy = np.mean(y == y_pred)
  return accuracy
print(f"Accuracy is: {accuracy(y_test, y_pred) * 100:.2f}%")

--- Random Forest using NumPy ---
Prediction: [0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 1 1 1 1]
y values: [0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 1 1 1 1]
Accuracy is: 99.27%


In [21]:
print("--- Random Forest using Scikit-Learn ---")

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


model_sk = RandomForestClassifier()
model_sk.fit(X_train_scaled, y_train)
y_pred = model_sk.predict(X_test_scaled)


print("Prediction:", y_pred[:20])
print("y values:", y_test[:20])
print(f"Accuracy is: {accuracy_score(y_test, y_pred) * 100:.2f}%")

--- Random Forest using Scikit-Learn ---
Prediction: [0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 1 1 1 1]
y values: [0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 1 1 1 1]
Accuracy is: 99.27%
